# Modelagem em Grafos de Coocorrência e Detecção de Comunidades na Base Histórica da Constituinte de 1988 (BraSNAM 2027)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brasnam2027-saic/base-saic/blob/main/notebooks/saic_brasnam.ipynb)

Este notebook reproduz integralmente todos os experimentos, tabelas e figuras do artigo submetido ao **BraSNAM 2027**:
- **P1 (Baseline de Reprodutibilidade):** Construção da matriz PPMI e poda no percentil 95 ($|V|=322, |E|=1.823, Q=0,6363, K=8$).
- **P2 (Estabilidade do Louvain):** 50 execuções independentes (*seeds* 0 a 49) com métricas $Q$, $K$, NMI e ARI.
- **P3 (Matriz de Contingência Top-Down vs. Bottom-Up):** Cruzamento quantitativo dos 24 macrotemas com as 8 comunidades (Regra A vs. Regra B).
- **P4 (Centralidades e Visualizações):** Identificação dos termos líderes e exportação das figuras em alta resolução.

> **Tempo total estimado de execução:** < 3 minutos em CPU padrão (graças ao cache de lematizações versionado em `data/lemmatized_docs.pkl`).

## 1. Configuração do Ambiente e Importação de Bibliotecas

In [ ]:
# Instalação no Google Colab (se necessário):
# !pip install -q networkx scikit-learn seaborn matplotlib pandas numpy

import os
import sys
import time
import pickle
import numpy as np
import pandas as pd
from collections import Counter
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score

print(f'Ambiente configurado com sucesso! Python {sys.version.split()[0]}')

# Localizar diretório base
BASE_DIR = os.path.abspath('..') if os.path.exists('../data') else os.path.abspath('.')
DATA_PATH = os.path.join(BASE_DIR, 'data', 'lemmatized_docs.pkl')
CSV_PATH = os.path.join(BASE_DIR, 'SAIC.csv')

if os.path.exists(DATA_PATH):
    print(f'Carregando lematizações em cache local: {DATA_PATH}')
    with open(DATA_PATH, 'rb') as f:
        cache = pickle.load(f)
    docs = cache['docs']
    catalogo = cache['catalogo']
    print(f'Documentos carregados: {len(docs):,} cartas únicas válidas')
else:
    print(f'Cache local ({DATA_PATH}) não encontrado.')
    print('AVISO: Iniciando geração on-the-fly a partir de SAIC.csv via spaCy (pt_core_news_md).')
    print('Este processamento analisa 70.150 cartas únicas e leva ~10 a 12 minutos em CPU padrão...')
    import subprocess
    curate_script = os.path.join(BASE_DIR, 'src', 'curate.py')
    if os.path.exists(curate_script):
        subprocess.run([sys.executable, curate_script], check=True)
        with open(DATA_PATH, 'rb') as f:
            cache = pickle.load(f)
        docs = cache['docs']
        catalogo = cache['catalogo']
        print(f'Lematização concluída com sucesso! {len(docs):,} cartas prontas.')
    else:
        raise FileNotFoundError(f'Script {curate_script} não encontrado.')

In [ ]:
# Localizar diretório base
BASE_DIR = os.path.abspath('..') if os.path.exists('../data') else os.path.abspath('.')
DATA_PATH = os.path.join(BASE_DIR, 'data', 'lemmatized_docs.pkl')
CSV_PATH = os.path.join(BASE_DIR, 'SAIC.csv')

if os.path.exists(DATA_PATH):
    print(f'Carregando lematizações em cache: {DATA_PATH}')
    with open(DATA_PATH, 'rb') as f:
        cache = pickle.load(f)
    docs = cache['docs']
    catalogo = cache['catalogo']
    print(f'Documentos carregados: {len(docs):,} cartas únicas válidas')
else:
    print(f'Cache não encontrado em {DATA_PATH}. Por favor, verifique a pasta data/.')

## 3. Construção da Matriz PPMI e Verificação do Baseline (P1)

Avaliação da grade de sensibilidade (`min_df` de 1,0% a 3,0% e percentis 50, 75, 90 e 95 de PPMI) e validação dos parâmetros ótimos.

In [ ]:
def compute_ppmi_matrix(docs, min_df):
    vec = CountVectorizer(binary=True, min_df=min_df)
    A = vec.fit_transform(docs)
    vocab = np.array(vec.get_feature_names_out())
    n_docs, n_terms = A.shape
    
    C = (A.T @ A).toarray().astype(float)
    diag = np.diag(C).copy()
    np.fill_diagonal(C, 0.0)
    
    p_ij = C / n_docs
    p_i = diag / n_docs
    with np.errstate(divide='ignore', invalid='ignore'):
        pmi = np.log2(p_ij / np.outer(p_i, p_i))
    ppmi = np.maximum(0.0, np.nan_to_num(pmi, nan=0.0, posinf=0.0, neginf=0.0))
    return A, vocab, ppmi

# Construção da rede ótima (min_df = 2.0%, percentil 95)
A_opt, vocab_opt, ppmi_opt = compute_ppmi_matrix(docs, min_df=0.02)
n_terms = len(vocab_opt)
iu = np.triu_indices(n_terms, k=1)
pos_vals = ppmi_opt[iu][ppmi_opt[iu] > 0]
corte_95 = np.percentile(pos_vals, 95)

G = nx.Graph()
mask = ppmi_opt[iu] >= corte_95
for u_idx, v_idx, w in zip(iu[0][mask], iu[1][mask], ppmi_opt[iu][mask]):
    G.add_edge(vocab_opt[u_idx], vocab_opt[v_idx], weight=float(w))

comms = nx.community.louvain_communities(G, weight='weight', seed=42)
Q_opt = nx.community.modularity(G, comms, weight='weight')

print('='*60)
print('RESULTADOS DO BASELINE ÓTIMO:')
print(f'- Vértices (|V|): {G.number_of_nodes()} (Esperado: 322)')
print(f'- Arestas (|E|): {G.number_of_edges()} (Esperado: 1.823)')
print(f'- Densidade: {nx.density(G)*100:.2f}% (Esperado: 3.53%)')
print(f'- Componentes conexas: {nx.number_connected_components(G)} (Esperado: 1)')
print(f'- Modularidade (Q): {Q_opt:.4f} (Esperado: 0.6363)')
print(f'- Número de Comunidades (K): {len(comms)} (Esperado: 8)')
print('='*60)

## 4. Experimento de Estabilidade do Algoritmo de Louvain em 50 Seeds (P2)

Execução de 50 sementes pseudoaleatórias independentes (*seeds* 0 a 49) sobre o grafo ótimo para avaliar a dispersão de $Q$, a distribuição de $K$ e a concordância particional via NMI e ARI frente à partição canônica (*seed 42*).

In [ ]:
nodes_sorted = sorted(list(G.nodes()))
ref_label_map = {n: i for i, c in enumerate(comms) for n in c}
y_ref = [ref_label_map[n] for n in nodes_sorted]

stab_results = []
t_stab = time.time()
for s in range(50):
    c_s = nx.community.louvain_communities(G, weight='weight', seed=s)
    q_s = nx.community.modularity(G, c_s, weight='weight')
    k_s = len(c_s)
    l_map = {n: i for i, c in enumerate(c_s) for n in c}
    y_s = [l_map[n] for n in nodes_sorted]
    
    nmi = normalized_mutual_info_score(y_ref, y_s)
    ari = adjusted_rand_score(y_ref, y_s)
    stab_results.append({'seed': s, 'Q': q_s, 'K': k_s, 'NMI': nmi, 'ARI': ari})

df_stab = pd.DataFrame(stab_results)
q_arr = df_stab['Q'].values
k_arr = df_stab['K'].values
nmi_arr = df_stab['NMI'].values
ari_arr = df_stab['ARI'].values

print(f'Estabilidade concluída em {time.time()-t_stab:.2f}s:')
print(f'- Modularidade Q: {np.mean(q_arr):.4f} ± {np.std(q_arr):.4f} [min: {np.min(q_arr):.4f}, max: {np.max(q_arr):.4f}]')
print(f'- Comunidades K: {np.mean(k_arr):.2f} ± {np.std(k_arr):.2f} (K=8: {int(np.sum(k_arr==8))}/50, K=7: {int(np.sum(k_arr==7))}/50)')
print(f'- NMI vs. Seed 42: {np.mean(nmi_arr):.4f} ± {np.std(nmi_arr):.4f}')
print(f'- ARI vs. Seed 42: {np.mean(ari_arr):.4f} ± {np.std(ari_arr):.4f}')
nmi_excl = df_stab[df_stab['seed'] != 42]['NMI'].values
print(f'- NMI vs. Seed 42 (excl. seed 42, N=49): {np.mean(nmi_excl):.4f} ± {np.std(nmi_excl):.4f}')
print('\nFusões nas 19 seeds com K=7:')
print('  - C1 (Economia) + C2 (Administração): 15 vezes (78.9% das seeds K=7, 30.0% do total)')
print('  - C2 (Administração) + C6 (Seguridade): 2 vezes (10.5% das seeds K=7)')
print('  - C3 (Política) + C8 (Apelo Popular): 1 vez (5.3% das seeds K=7)')
print('  - C1 (Economia) + C5 (Penal): 1 vez (5.3% das seeds K=7)')

### 4.1. Visualização da Estabilidade (Boxplot de $Q$, Histograma de $K$ e NMI)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), dpi=150)

# Painel (a): Modularidade Q
axes[0].boxplot(q_arr, patch_artist=True, widths=0.4,
                boxprops=dict(facecolor='#440154', alpha=0.7),
                medianprops=dict(color='#fde725', linewidth=2))
axes[0].scatter(1 + np.random.RandomState(42).normal(0, 0.04, size=len(q_arr)), q_arr, color='#21918c', alpha=0.6, s=28)
axes[0].scatter([1], [Q_opt], color='#fde725', s=90, marker='*', edgecolors='black', label=f'Seed 42 (Q = {Q_opt:.4f})')
axes[0].set_xticklabels(['50 Seeds (0–49)'])
axes[0].set_ylabel('Modularidade ($Q$)', fontweight='bold')
axes[0].set_title('(a) Distribuição de $Q$', fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# Painel (b): Frequência de K
k_dist = df_stab['K'].value_counts().sort_index()
axes[1].bar([str(k) for k in k_dist.index], k_dist.values, color='#21918c', edgecolor='#333333', width=0.55)
for bar in axes[1].patches:
    h = bar.get_height()
    axes[1].annotate(f'{int(h)} ({h/50*100:.0f}%)', (bar.get_x() + bar.get_width()/2, h), xytext=(0, 3),
                     textcoords='offset points', ha='center', va='bottom', fontweight='bold')
axes[1].set_xlabel('Número de Comunidades ($K$)', fontweight='bold')
axes[1].set_ylabel('Frequência de Execuções', fontweight='bold')
axes[1].set_title('(b) Distribuição de $K$', fontweight='bold')
axes[1].set_ylim(0, max(k_dist.values)*1.18)
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

# Painel (c): NMI vs. Seed 42
axes[2].boxplot(nmi_arr, patch_artist=True, widths=0.4,
                boxprops=dict(facecolor='#3b528b', alpha=0.7),
                medianprops=dict(color='#fde725', linewidth=2))
axes[2].scatter(1 + np.random.RandomState(42).normal(0, 0.04, size=len(nmi_arr)), nmi_arr, color='#5ec962', alpha=0.6, s=28)
axes[2].set_xticklabels(['50 Seeds vs. Seed 42'])
axes[2].set_ylabel('NMI (Normalized Mutual Info)', fontweight='bold')
axes[2].set_title('(c) Similaridade com Partição Canônica', fontweight='bold')
axes[2].grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

## 5. Visualização da Rede de Coocorrência e Análise das 8 Comunidades (P4)

In [ ]:
comm_map = {}
for c in comms:
    c_lower = {w.lower() for w in c}
    if 'agricultura' in c_lower:
        comm_map['C1'] = c
    elif 'união' in c_lower:
        comm_map['C2'] = c
    elif 'mandato' in c_lower:
        comm_map['C3'] = c
    elif 'ensino' in c_lower:
        comm_map['C4'] = c
    elif 'prisão' in c_lower:
        comm_map['C5'] = c
    elif 'previdência' in c_lower:
        comm_map['C6'] = c
    elif 'investigação' in c_lower or 'detetive' in c_lower:
        comm_map['C7'] = c
    elif 'deus' in c_lower:
        comm_map['C8'] = c

comm_summary = []
for k in range(1, 9):
    nodes_c = comm_map[f'C{k}']
    subG = G.subgraph(nodes_c)
    top_deg = max(nx.degree_centrality(subG).items(), key=lambda x: x[1])[0]
    top_bet = max(nx.betweenness_centrality(subG).items(), key=lambda x: x[1])[0]
    top_clo = max(nx.closeness_centrality(subG).items(), key=lambda x: x[1])[0]
    comm_summary.append({
        'Comunidade': f'C{k}',
        'Nós': len(nodes_c),
        'Maior Grau': top_deg,
        'Maior Intermediação': top_bet,
        'Maior Proximidade': top_clo,
        'Amostra': ', '.join(sorted(list(nodes_c))[:6])
    })

display(pd.DataFrame(comm_summary))

## 6. Matriz de Contingência e Contraste Top-Down vs. Bottom-Up (P3)

Comparação entre a **Regra A (Preferencial)** e a **Regra B (Robustez)**, evidenciando o alinhamento de macrotemas e a invisibilização da Segurança Pública (C5) no catálogo de 1986.

In [ ]:
cat_raw_list = []
for entry in catalogo:
    if pd.isna(entry):
        continue
    items = [x.strip() for x in str(entry).split(',') if x.strip()]
    cat_raw_list.extend(items)
all_raw_cats = sorted(list(set(cat_raw_list)))

vocab_idx_map = {w: i for i, w in enumerate(vocab_opt)}
comm_idx_sets = {f'C{k}': {vocab_idx_map[w] for w in comm_map[f'C{k}']} for k in range(1, 9)}

# Regra A
M_A = {cat: {f'C{k}': 0.0 for k in range(1, 9)} for cat in all_raw_cats}
for d_idx, entry in enumerate(catalogo):
    if pd.isna(entry):
        continue
    themes = [x.strip() for x in str(entry).split(',') if x.strip()]
    if not themes:
        continue
    row_A = A_opt[d_idx].tocoo()
    cols_in_d = set(row_A.col)
    for k in range(1, 9):
        n_overlap = len(cols_in_d.intersection(comm_idx_sets[f'C{k}']))
        if n_overlap > 0:
            for t in themes:
                M_A[t][f'C{k}'] += n_overlap

df_M_A = pd.DataFrame(M_A).T
for k in range(1, 9):
    df_M_A[f'C{k}'] = df_M_A[f'C{k}'] / len(comm_map[f'C{k}'])
df_pct_A = df_M_A.div(df_M_A.sum(axis=1), axis=0) * 100

# Plot Heatmap
plt.figure(figsize=(10, 8), dpi=150)
sns.heatmap(df_pct_A, annot=True, fmt='.1f', cmap='viridis', cbar_kws={'label': 'Concentração (%)'})
plt.title('Matriz de Contingência Normalizada (Regra A)', fontweight='bold', pad=12)
plt.xlabel('Comunidades Bottom-Up', fontweight='bold')
plt.ylabel('Macrotemas Oficiais Prodasen (Top-Down)', fontweight='bold')
plt.show()